In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

Getting the atmospheric parameters dataset for the selected stars: 

In [8]:
df = pd.read_csv("cflib_params_Wu11 (1).txt")

In [9]:
df.head()

,star,T_eff,log_g,metallicity
0,HD100889.fits,10563.0,3.83,-0.27
1,HD102212.fits,3738.0,1.55,-0.41
2,HD102224.fits,4462.0,2.05,-0.38
3,HD102328.fits,4389.0,2.59,0.30
4,HD102870.fits,6071.0,4.05,0.13


Initializing the RF interpolator

In [4]:
from sklearn.externals import joblib

In [5]:
INTERPOLATOR_MODEL = '../../models/RF_base_model_pvk.sav'
STANDARD_SCALER = '../../models/rf_standard_scale_pvk.sav'

In [6]:
sc = joblib.load(STANDARD_SCALER)
ip = joblib.load(INTERPOLATOR_MODEL)

/home/arbiter/projects/Recalibration-deep-learning/.env27/lib/python2.7/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator StandardScaler from version 0.20.2 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
/home/arbiter/projects/Recalibration-deep-learning/.env27/lib/python2.7/site-packages/sklearn/ensemble/weight_boosting.py:29: DeprecationWarning: numpy.core.umath_tests is an internal NumPy module and should not be imported. It will be removed in a future NumPy release.
  from numpy.core.umath_tests import inner1d
/home/arbiter/projects/Recalibration-deep-learning/.env27/lib/python2.7/site-packages/sklearn/base.py:311: UserWarning: Trying to unpickle estimator DecisionTreeRegressor from version 0.20.2 when using version 0.19.1. This might lead to breaking code or invalid results. Use at your own risk.
  UserWarning)
/home/arbiter/projects/Recalibration-deep-learning/.env27/lib/python2.7/sit

In [7]:
ip.n_jobs = 1 

def InterpolateSpectrum(temperature, gravity, metallicity):
    X = np.array([temperature, gravity, metallicity]).reshape(1, -1)
    X_std = sc.transform(X)
    y = ip.predict(X_std)
    return y

### Interpolating for each datapoint in the dataset

Interpolating for all the selected stars and creating a file containing the array of all the interpolated flux : 

In [29]:
flux_arr = []

for i in range(len(df)):
    pred = InterpolateSpectrum(temperature= df.loc[i][1], gravity= df.loc[i][2], metallicity= df.loc[i][3])
    flux_arr.append(pred[0,:])

flux = np.array(flux_arr)

print(flux)
print(len(flux))

[[1.18601767 1.22942634 1.17626099 ... 0.42062923 0.42171141 0.42278666]
 [0.09260631 0.10392712 0.1085579  ... 1.89134097 1.89760813 1.91798774]
 [0.12836964 0.1376427  0.13979537 ... 0.82752983 0.81743293 0.82009242]
 ...
 [0.78356789 0.76333651 0.77973124 ... 0.6061509  0.58566458 0.58417524]
 [0.23961371 0.32799682 0.33880731 ... 0.6996219  0.69037811 0.68241546]
 [0.69764983 0.75644742 0.76383109 ... 0.62934402 0.62847065 0.63141824]]
850


Saving the flux data into a file

In [ ]:
# np.save("rf_interpolated_fluxes", flux)